# Robinhood Day Trading Bot — Dry Run Test

This notebook runs the trading bot in **DRY RUN mode** — it will:
- ✅ Log into your Robinhood account
- ✅ Scan for stocks with momentum signals
- ✅ Show you the signal list and ask which ones you want to buy
- ❌ NOT place any real orders (DRY_RUN = True)

Run each cell from top to bottom. Click the play button ▶ on the left of each cell.

In [ ]:
# STEP 1 — Install dependencies (takes ~1 minute)
!pip install robin_stocks yfinance pandas_ta pandas numpy pytz finvizfinance requests -q

In [ ]:
# STEP 2 — Enter your Robinhood credentials
# These are only stored in this session and never saved anywhere
import os
import getpass

os.environ['ROBINHOOD_USERNAME'] = input('Enter your Robinhood email: ')
os.environ['ROBINHOOD_PASSWORD'] = getpass.getpass('Enter your Robinhood password (hidden): ')
print('Credentials set.')

In [ ]:
# STEP 3 — Download the trading script
import os, sys, urllib.request

# Download trader.py directly from GitHub
url = 'https://raw.githubusercontent.com/nrienks23-sketch/Robinhood/claude/dreamy-noether-4u7oj3/trader.py'
dest = '/content/trader.py'

urllib.request.urlretrieve(url, dest)

if not os.path.exists(dest):
    raise Exception('Download failed — check your internet connection.')

sys.path.insert(0, '/content')
os.chdir('/content')
print('Script loaded. trader.py is ready.')

In [ ]:
# STEP 4 — Run a single scan and pick which stocks to buy
# This will log into Robinhood (may ask for 2FA code),
# scan for signals, print the ranked list, then ask YOU which ones to buy.

import sys, os, importlib

# Make sure Python can find trader.py
for path in ['/content', '/content/Robinhood', '.']:
    if os.path.exists(os.path.join(path, 'trader.py')):
        if path not in sys.path:
            sys.path.insert(0, path)
        os.chdir(path)
        break

import trader
importlib.reload(trader)

assert trader.DRY_RUN == True, 'DRY_RUN must be True for this test!'

logged_in = trader.rh_login()

if logged_in:
    print('\n✅ Logged into Robinhood successfully!\n')
    positions = trader.load_positions()

    print('Scanning for entry signals... (this takes a few minutes)\n')
    entries = trader.scan_for_entries(positions)
    trader.print_summary(positions, entries)

    # Build list of buyable tickers (6/6 and 5/6, not already held)
    buyable = [
        (score, ticker, price, detail)
        for score in [6, 5]
        for ticker, price, detail in entries.get(score, [])
        if ticker not in positions
    ]

    if buyable and len(positions) < trader.MAX_POSITIONS:
        print('\nEnter ticker(s) to BUY (comma-separated), or press Enter to skip:')
        print(f'Available slots: {trader.MAX_POSITIONS - len(positions)} / {trader.MAX_POSITIONS}')
        raw = input('> ').strip().upper()
        if raw:
            chosen = [t.strip() for t in raw.split(',') if t.strip()]
            buyable_map = {ticker: (price, detail) for _, ticker, price, detail in buyable}
            for ticker in chosen:
                if len(positions) >= trader.MAX_POSITIONS:
                    print('Max positions reached.')
                    break
                if ticker in buyable_map:
                    price, detail = buyable_map[ticker]
                    trader.enter_position(ticker, price, positions)
                else:
                    print(f'{ticker} not in signal list — skipping.')
        else:
            print('No buys selected.')
    elif not buyable:
        print('No 5/6 or 6/6 signals to act on right now.')
else:
    print('❌ Login failed. Check your username/password and try again.')

In [ ]:
# STEP 5 — Monitor open positions (runs until you stop it)
# Checks every 30 seconds for stop-loss / tier exits on anything you bought.
# Press the STOP button (■) in the toolbar to stop early.

import time

print('Monitoring positions. Press STOP to exit.\n')
while True:
    trader.manage_positions(positions)
    trader.print_summary(positions, {})
    print('Waiting 30 seconds...\n')
    time.sleep(30)